# PopOut AI - Complete Project Report

This notebook is an extensive English report for the PopOut AI project. It consolidates the implementation, algorithmic design, datasets, experimental evidence, and head-to-head comparisons between the implemented agents.

The project implements PopOut, a Connect Four variant where each move can either drop a piece into a column or pop one of the current player's bottom pieces out of a column. The work covers adversarial search, optimized search, a custom ID3 decision tree, dataset generation from MCTS, and playable interfaces.


## Executive Summary

The project satisfies the main assignment requirements:

| Requirement | Project evidence |
|---|---|
| Playable PopOut program | CLI and Pygame GUI support Human vs Human, Human vs Computer, and Computer vs Computer. |
| MCTS with UCT | `StandardUCT` implements the classical select, expand, simulate, backpropagate loop with UCT selection. |
| MCTS variants | `ExperimentalUCT`, `SolverMCTS`, reuse engines, and Numba-optimized engines are implemented. |
| Decision tree learned with ID3 | `ID3Classifier` is implemented from scratch in `src/decision_tree/id3/learner.py`; no library decision-tree trainer is used. |
| Iris warm-up | The Iris notebook discretizes continuous features and evaluates custom ID3 with repeated cross-validation. |
| PopOut dataset generated from MCTS | The PopOut dataset is generated by MCTS/solver self-play and labeled as `(state, best_move)`. |
| Computer vs Computer comparison | Multiple tournaments compare MCTS, solver MCTS, optimized engines, ID3 with tactical features, and raw ID3. |

Overall, the strongest parts of the work are the compact bitboard engine, the breadth of MCTS variants, the optimized Numba engines, the custom ID3 pipeline, and the empirical comparison notebooks. The main caveat is environment reproducibility: the optimized engines require the Conda environment with Python 3.10 and Numba.


In [9]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import json
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "generated"
FIG_DIR = ROOT / "data" / "figures"
print(f"Project root: {ROOT}")


Project root: /home/jose-paulo/Nova pasta/PopoutAI/popout-ai


## 1. Project Architecture

The implementation is organized as a Python package under `src/`:

| Area | Main files | Role |
|---|---|---|
| Game engine | `src/engine/standard/bitboard.py`, `src/engine/standard/rules.py` | Board representation, legal moves, drop/pop mechanics, win detection, draw/repetition helpers. |
| Optimized engine | `src/engine/optimized/numba_bitboard.py`, `src/engine/optimized/numba_rules.py` | Numba-compatible board/rule kernels. |
| MCTS | `src/mcts/standard/base.py`, `uct_standard.py`, `uct_experimental.py`, `uct_solver.py`, `uct_reuse.py` | Standard UCT, experimental UCT, solver-style proof propagation, and tree reuse. |
| Optimized MCTS | `src/mcts/optimized/numba_mcts.py`, `numba_solver.py`, `numba_reuse.py`, `numba_search.py` | JIT-accelerated search loops and solver engines. |
| Decision tree | `src/decision_tree/id3/learner.py`, `id3_agent.py`, `id3_agent_raw.py`, `dataset_generator.py` | Custom ID3, PopOut dataset generation, playable tree-based agents. |
| Interfaces | `src/interfaces/cli.py`, `src/interfaces/gui/` | Human and AI game modes. |
| Validation | `tests/` | Rule, MCTS, ID3, Numba, GUI, CLI, integration, and performance tests. |

The architecture separates rules from search algorithms, and search algorithms from interfaces. That makes it possible to run the same engines in the CLI, GUI, tests, notebooks, and dataset generator.


## 2. Game Engine Analysis

The board is represented with two integer bitmasks: one for player 1 and one for player 2. The 7 by 6 board is stored as 7 columns of 7 bits each: 6 playable bits plus one guard bit. This guard bit prevents false horizontal/diagonal connections when using bit shifts.

Move encoding is compact:

| Move integer | Meaning |
|---:|---|
| `0..6` | Drop in column `0..6`. |
| `7..13` | Pop from column `0..6`. |

Important engine details:

- `legal_moves()` returns all valid drops plus valid pops for the current player.
- `apply_move()` mutates the bitboards and switches the current player.
- `has_won(mask)` detects four-in-a-row with constant-time bitwise shifts.
- `evaluate_after_move(board, mover)` implements the PopOut simultaneous-win rule: if a pop creates winning lines for both players, the mover wins.
- `extended_features(board)` adds tactical features for ID3: threats, center control, phase, immediate win availability, and opponent immediate-win danger.

This is a technically strong design because the critical operations used by MCTS are O(1) and avoid slow nested list traversal.


In [10]:
from src.engine.standard.bitboard import PopOutBoard
from src.engine.standard.rules import has_won, evaluate_after_move

board = PopOutBoard()
for move in [3, 3, 2, 2, 1, 1]:
    board.apply_move(move)
print(board)
print("Legal moves:", board.legal_moves())


. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. O O O . . .
. X X X . . .
0 1 2 3 4 5 6
Legal moves: [0, 1, 2, 3, 4, 5, 6, 8, 9, 10]


## 3. MCTS Algorithms

The baseline MCTS implementation follows the four canonical phases:

1. Selection: descend the current tree using UCT.
2. Expansion: add one untried legal move as a child node.
3. Simulation: run a rollout until win, draw, repetition, or depth limit.
4. Backpropagation: update visit counts and values along the path.

The UCT score is:

$$
UCT_i = \bar{x}_i + C \sqrt{\frac{\ln N}{n_i}}
$$

where `x_i` is the average reward, `N` is parent visits, `n_i` is child visits, and `C` is the exploration constant.

Implemented MCTS family:

| Agent | Main idea | Notes |
|---|---|---|
| `StandardUCT` | Classical UCT MCTS | Baseline adversarial search. |
| `ExperimentalUCT` | Alternative handling of unvisited children | Useful for comparing selection behavior. |
| `SolverMCTS` | MCTS plus proof statuses | Propagates proven WIN, LOSS, DRAW and minimax distance. |
| `ReuseUCT` | Reuses the surviving subtree between turns | Reduces wasted search after predictable continuations. |
| `NumbaMCTS` | Python tree with Numba rollout kernels | Faster simulations. |
| `FlatNumbaMCTS` | Full flat-array MCTS loop inside Numba | Highest plain-MCTS throughput. |
| `NumbaSolverMCTS` | Optimized solver-style MCTS | Combines proof logic and JIT kernels. |
| `FlatNumbaSolverMCTS` | Flat-array optimized solver | Fast solver variant, used as a strong oracle. |
| `ReuseNumbaMCTS`, `ReuseFlatNumbaSolverMCTS` | Reuse variants for optimized engines | Designed for repeated game play rather than isolated single decisions. |


### 3.1 MCTS-Solver

`SolverMCTS` goes beyond probabilistic MCTS by assigning proof statuses to nodes:

| Status | Meaning |
|---|---|
| `UNKNOWN` | Outcome has not been proven; use normal MCTS statistics. |
| `WIN` | The player to move has a forced win. |
| `LOSS` | The player to move is in a forced loss. |
| `DRAW` | The player to move can force at least a draw. |

The key logic is an AND/OR proof tree:

- A node is a proven WIN if at least one child is a proven LOSS for the opponent.
- A node is a proven LOSS if all children are proven WIN for the opponent.
- A node is a proven DRAW if the best achievable proven outcome is a draw.

The solver also stores minimax distance: it prefers faster wins and slower losses. This fixes a limitation of ordinary MCTS, where a one-move win and a five-move win can both look like reward 1.0.


## 4. Decision Tree and ID3

The ID3 implementation is custom and located in `src/decision_tree/id3/learner.py`.

Core methods:

| Method | Purpose |
|---|---|
| `entropy(labels)` | Shannon entropy of target labels. |
| `information_gain(df, feature, target)` | Reduction in entropy from splitting by a feature. |
| `build_tree(...)` | Recursive ID3 tree construction. |
| `predict_one(row)` | Traverse the learned tree for one sample. |
| `score(df, target)` | Accuracy over a labeled dataset. |
| `get_feature_importance()` | Split-frequency based feature importance. |

The implementation handles pure leaves, no-feature fallbacks, maximum depth, unknown branch values via node majority labels, and validation of empty or invalid data.

Two PopOut ID3 agents exist:

| Agent | Features | Tactical safeguards | Intended interpretation |
|---|---|---|---|
| `ID3Agent` | Raw board cells plus tactical engineered features | Yes: immediate win and immediate block before tree prediction | Hybrid playable decision-tree agent. |
| `ID3AgentRaw` | Only raw board cells plus current player | No hard-coded tactical layer | More purely learned policy from board patterns. |

This distinction matters academically: classifier accuracy measures the tree, while game strength of `ID3Agent` measures a hybrid tree-plus-tactics player.


### 4.1 Iris Warm-Up Results

The Iris notebook evaluates the custom ID3 classifier on a standard categorical version of the Iris problem. Continuous features are discretized using quantile bins fitted on training data to avoid leakage.

Saved notebook results:

| Metric | Value |
|---|---:|
| Repeated folds | 50 evaluations, 5 repetitions times 10 folds |
| Mean accuracy | 95.07% |
| Standard deviation | 5.63% |
| Minimum fold accuracy | 80.00% |
| Maximum fold accuracy | 100.00% |
| F1, Iris-setosa | 0.986 |
| F1, Iris-versicolor | 0.925 |
| F1, Iris-virginica | 0.941 |

These results validate the custom ID3 implementation on a known supervised-learning task before applying it to PopOut.


## 5. PopOut Dataset Analysis

The PopOut dataset is generated from MCTS/solver play. Each row stores a board state and the best move selected by the oracle.

The project contains two main generated datasets:

- v1: 3,000 games, 100k oracle iterations.
- v2: 5,000 games, 100k oracle iterations.

The v2 generator uses two opponent modes: oracle-with-blunders and blocking-random. It also mirrors positions horizontally, which doubles useful data because PopOut is symmetric left-to-right.


In [11]:
def summarize_dataset(path):
    df = pd.read_csv(path)
    move_counts = df["best_move"].value_counts() if "best_move" in df else pd.Series(dtype=int)
    summary = {
        "path": str(path.relative_to(ROOT)),
        "rows": len(df),
        "columns": len(df.columns),
        "drop_labels": int(df["best_move"].str.startswith("drop").sum()) if "best_move" in df else None,
        "pop_labels": int(df["best_move"].str.startswith("pop").sum()) if "best_move" in df else None,
        "proven_rows": int(df["is_proven"].sum()) if "is_proven" in df else None,
        "proven_share": float(df["is_proven"].mean()) if "is_proven" in df else None,
        "top_move": move_counts.index[0] if not move_counts.empty else None,
        "top_move_count": int(move_counts.iloc[0]) if not move_counts.empty else None,
    }
    return summary

dataset_paths = [
    DATA_DIR / "v1_3000games_100k" / "popout_dt_dataset.csv",
    DATA_DIR / "v2_5000games_100k" / "popout_dt_dataset.csv",
    DATA_DIR / "uct_standard.csv",
]

pd.DataFrame([summarize_dataset(p) for p in dataset_paths if p.exists()])


,path,rows,columns,drop_labels,pop_labels,proven_rows,proven_share,top_move,top_move_count
0,data/generated/v1_3000games_100k/popout_dt_dat...,119676,52,109896,9780,57272.0,0.478559,drop_3,27576
1,data/generated/v2_5000games_100k/popout_dt_dat...,155494,52,144830,10664,67306.0,0.432853,drop_3,46530
2,data/generated/uct_standard.csv,200,44,193,7,NaN,NaN,drop_3,54


Observed dataset statistics from the current repository:

| Dataset | Rows | Columns | Drop labels | Pop labels | Proven rows | Proven share | Most frequent move |
|---|---:|---:|---:|---:|---:|---:|---|
| v1 3k games, 100k oracle | 119,676 | 52 | 109,896 | 9,780 | 57,272 | 47.9% | `drop_3` |
| v2 5k games, 100k oracle | 155,494 | 52 | 144,830 | 10,664 | 67,306 | 43.3% | `drop_3` |
| legacy `uct_standard.csv` | 200 | 44 | 193 | 7 | n/a | n/a | `drop_3` |

Interpretation:

- Central drops dominate, especially `drop_3`, which matches the known strategic value of center control in Connect Four-like games.
- Pop labels are much rarer than drop labels: 8.2% in v1 and 6.9% in v2. This class imbalance is expected because popping is legal only when the current player owns the bottom cell and is often tactically risky.
- Proven rows are a major asset: they are labels derived from solver proof status rather than only stochastic estimates.


In [12]:
# Detailed v2 label and tactical-feature distribution.
v2_path = DATA_DIR / "v2_5000games_100k" / "popout_dt_dataset.csv"
if v2_path.exists():
    df_v2 = pd.read_csv(v2_path)
    display(df_v2["best_move"].value_counts().rename_axis("move").reset_index(name="count").head(14))
    display(pd.DataFrame({
        "current_player": df_v2["current_player"].value_counts().sort_index(),
        "phase": df_v2["phase"].value_counts().sort_index(),
    }))
    print("can_win distribution:", df_v2["can_win"].value_counts().sort_index().to_dict())
    print("opp_wins_next distribution:", df_v2["opp_wins_next"].value_counts().sort_index().to_dict())
else:
    print("v2 dataset not found")


,move,count
0,drop_3,46530
1,drop_4,27079
2,drop_2,27079
3,drop_1,16345
4,drop_5,16345
5,drop_0,5726
6,drop_6,5726
7,pop_3,2696
8,pop_2,1942
9,pop_4,1942


,current_player,phase
0,NaN,91680
1,81122.0,62354
2,74372.0,1460


can_win distribution: {0: 145774, 1: 9720}
opp_wins_next distribution: {0: 132162, 1: 23332}


![PopOut dataset EDA](../data/figures/popout_dataset_eda.png)

![Preferred move heatmap](../data/figures/move_heatmap.png)


## 6. ID3 Training Results on PopOut

Saved PopOut decision-tree pipeline results for the v2 dataset:

| Model | Train accuracy | Test accuracy | Notes |
|---|---:|---:|---|
| ID3 with tactical features | 0.868 | 0.854 | Trained with 4x oversampling of proven rows. |
| ID3 with tactical features, earlier main notebook run | n/a | 0.855 | Consistent with the pipeline result. |

Tactical subset accuracy from the v2 pipeline:

| Subset | Test examples | Accuracy |
|---|---:|---:|
| `can_win = 1` | 2,338 | 0.813 |
| `opp_wins_next = 1` | 4,777 | 0.667 |
| Pop move labels | 1,806 | 0.778 |

The tree learns a strong imitation policy, but the hardest positions are tactical defense positions where the opponent threatens an immediate win. This explains why the playable `ID3Agent` adds immediate-win and immediate-block safeguards before consulting the tree.


![ID3 accuracy versus dataset size](../data/figures/id3_accuracy_vs_size.png)

![PopOut feature importance](../data/figures/popout_feature_importance.png)

![ID3 confusion matrix](../data/figures/id3_confusion_matrix.png)


## 7. Performance Analysis

The project evaluates performance in two complementary ways:

1. Search throughput: iterations per second for MCTS engines.
2. Decision quality: tournament results and win rate against random or other agents.

Saved technical benchmark results, 10,000 iterations, best of 3 runs:

| Engine | Time | Throughput | Relative observation |
|---|---:|---:|---|
| `StandardUCT` | 923 ms | 10,839 iter/s | Pure Python baseline. |
| `SolverMCTS` | 1,028 ms | 9,730 iter/s | Slightly slower due to proof bookkeeping. |
| `NumbaMCTS` | 148 ms | 67,709 iter/s | About 6x faster than baseline. |
| `NumbaSolverMCTS` | 198 ms | 50,477 iter/s | Solver plus JIT acceleration. |
| `FlatNumbaMCTS` | 44 ms | 225,569 iter/s | About 21x faster than baseline. |
| `FlatNumbaSolverMCTS` | 45 ms | 220,814 iter/s | Fast solver architecture. |

The main lesson is that the flat-array Numba design removes most Python overhead from the search loop. This is why the dataset generator can use very high oracle budgets.


![MCTS throughput](../data/figures/mcts_throughput.png)

![Numba throughput](../data/figures/numba_throughput.png)

![All engines throughput](../data/figures/all_engines_throughput.png)


### 7.1 Speed vs Quality

Saved main-notebook benchmark against a random agent:

| Agent | Average decision time | Win rate vs random |
|---|---:|---:|
| MCTS-50 | 4.5 ms | 93% |
| MCTS-300 | 27.0 ms | 93% |
| MCTS-1000 | 89.4 ms | 100% |
| MCTS-3000 | 270.9 ms | 100% |
| ID3 with tactical features | 1.9 ms | 100% |
| ID3 Raw | 2.2 ms | 67% |

Interpretation:

- More MCTS iterations generally improve reliability but increase decision time.
- ID3 with tactical features is the best speed-quality trade-off in this benchmark.
- Raw ID3 is fast but less reliable unless the dataset and tree depth compensate for the missing engineered features.


![Speed quality tradeoff](../data/figures/speed_quality_tradeoff.png)


## 8. Head-to-Head Algorithm Comparisons

This section gives exact game counts. The important methodological point is color alternation: PopOut shows a strong first-player advantage in several strong-agent settings. Therefore, a fair comparison must report both the overall winner counts and what happens when each algorithm starts first.


### 8.1 Saved Tournament Evidence from Existing Notebooks

| Match | Games | Budget | Result |
|---|---:|---:|---|
| StandardUCT vs ExperimentalUCT | 20 | 300 iter/move | ExperimentalUCT 13 wins, StandardUCT 7 wins, 0 draws. |
| StandardUCT vs ID3Agent | 50 | StandardUCT 500 iter/move | ID3Agent 27 wins, StandardUCT 23 wins, 0 draws. |
| ID3Agent vs ID3Raw | 20 | ID3 agents | ID3Agent 10 wins, ID3Raw 10 wins, 0 draws. All games were won by Player 1. |
| StandardUCT vs SolverMCTS | 10 | 1,000 iter/move | SolverMCTS 7 wins, StandardUCT 3 wins, 0 draws. |
| StandardUCT vs SolverMCTS | 10 | 5,000 iter/move | SolverMCTS 7 wins, StandardUCT 3 wins, 0 draws. |
| FlatNumbaSolverMCTS 10k vs FlatNumbaMCTS 100k | 10 | asymmetric | FlatNumbaMCTS 7 wins, FlatNumbaSolverMCTS 3 wins, 0 draws. |

Detailed saved StandardUCT vs ExperimentalUCT orientation from the 20-game notebook run:

| Orientation | StandardUCT wins | ExperimentalUCT wins | Draws |
|---|---:|---:|---:|
| StandardUCT as Player 1, ExperimentalUCT as Player 2 | 5 | 5 | 0 |
| ExperimentalUCT as Player 1, StandardUCT as Player 2 | 2 | 8 | 0 |
| Overall | 7 | 13 | 0 |

This shows that ExperimentalUCT performed especially well when it started first, while the Standard-first half was tied.


### 8.2 Local Reproducible Tournament Run

This section is designed so the evaluator can run the comparisons locally instead of trusting precomputed numbers.

By default, the execution flag in the next cell is set to `False` so that opening or running the full notebook does not unexpectedly spend several minutes in tournaments. To regenerate the results on your own machine, set:

```python
RUN_LOCAL_TOURNAMENTS = True
```

Then run the cell. The tournament runner will:

- alternate colours automatically: game 1 uses Algorithm A as Player 1, game 2 uses Algorithm B as Player 1, and so on;
- report overall wins, draws, Player 1 wins, and Player 2 wins;
- report an orientation breakdown: `A as P1 vs B as P2` and `B as P1 vs A as P2`;
- save the generated rows to `data/generated/local_tournament_results.csv`.

The default non-Numba configuration below works in a basic environment. For Numba agents such as `flat_numba`, `numba_solver`, or `flat_numba_solver`, activate the Conda environment from `environment.yml` first.


In [13]:
from collections import Counter, defaultdict
from contextlib import redirect_stdout
from io import StringIO
import time

from src.mcts.factory import get_agent
from src.engine.standard.bitboard import PopOutBoard
from src.engine.standard.rules import board_signature, evaluate_after_move, is_threefold_repetition

RUN_LOCAL_TOURNAMENTS = False  # Change to True to generate the report numbers locally.

DISPLAY = {
    "standard": "StandardUCT",
    "experimental": "ExperimentalUCT",
    "solver": "SolverMCTS",
    "id3": "ID3Agent",
    "id3_raw": "ID3Raw",
    "numba": "NumbaMCTS",
    "flat_numba": "FlatNumbaMCTS",
    "numba_solver": "NumbaSolverMCTS",
    "flat_numba_solver": "FlatNumbaSolverMCTS",
    "reuse": "ReuseUCT",
    "reuse_numba": "ReuseNumbaMCTS",
    "reuse_flat_numba_solver": "ReuseFlatNumbaSolverMCTS",
}

# Keep this modest for quick local reproduction. Increase games/iterations for stronger evidence.
LOCAL_MATCHES = [
    {"a": "standard", "b": "experimental", "games": 10, "iterations": {"standard": 300, "experimental": 300}},
    {"a": "standard", "b": "solver", "games": 10, "iterations": {"standard": 300, "solver": 300}},
    {"a": "standard", "b": "id3", "games": 10, "iterations": {"standard": 300, "id3": 0}},
    {"a": "id3", "b": "id3_raw", "games": 10, "iterations": {"id3": 0, "id3_raw": 0}},
]

def quiet_get_agent(key, seed):
    with redirect_stdout(StringIO()):
        return get_agent(key, seed=seed)

def quiet_run(agent, board, iterations):
    with redirect_stdout(StringIO()):
        return agent.run(board, iterations=iterations)

def play_local_game(p1_key, p2_key, iterations, seed=0, max_moves=180):
    board = PopOutBoard()
    history = []
    agents = {
        1: (quiet_get_agent(p1_key, seed * 2 + 1), p1_key),
        2: (quiet_get_agent(p2_key, seed * 2 + 2), p2_key),
    }

    for ply in range(max_moves):
        history.append(board_signature(board))
        if is_threefold_repetition(history):
            return {"winner_player": -1, "winner_algorithm": "Draw", "plies": ply, "reason": "threefold"}

        legal = board.legal_moves()
        if not legal:
            return {"winner_player": -1, "winner_algorithm": "Draw", "plies": ply, "reason": "no_legal"}

        agent, key = agents[board.current_player]
        move = quiet_run(agent, board.clone(), iterations.get(key, 0))
        if move not in legal:
            return {"winner_player": -1, "winner_algorithm": "Draw", "plies": ply, "reason": f"illegal_{key}_{move}"}

        mover = board.current_player
        board.apply_move(move)
        winner = evaluate_after_move(board, mover=mover)
        if winner:
            return {"winner_player": winner, "winner_algorithm": agents[winner][1], "plies": ply + 1, "reason": "win"}

    return {"winner_player": -1, "winner_algorithm": "Draw", "plies": max_moves, "reason": "max_moves"}

def run_local_match(agent_a, agent_b, games=10, iterations=None, seed0=5000):
    iterations = iterations or {}
    rows = []
    for i in range(games):
        p1, p2 = (agent_a, agent_b) if i % 2 == 0 else (agent_b, agent_a)
        out = play_local_game(p1, p2, iterations, seed=seed0 + i)
        rows.append({
            "game": i + 1,
            "algorithm_A": DISPLAY.get(agent_a, agent_a),
            "algorithm_B": DISPLAY.get(agent_b, agent_b),
            "player1_algorithm": DISPLAY.get(p1, p1),
            "player2_algorithm": DISPLAY.get(p2, p2),
            "winner_player": out["winner_player"],
            "winner_algorithm": DISPLAY.get(out["winner_algorithm"], out["winner_algorithm"]),
            "plies": out["plies"],
            "reason": out["reason"],
        })
    return pd.DataFrame(rows)

def summarize_local_tournaments(df):
    overall = (
        df.groupby(["algorithm_A", "algorithm_B", "winner_algorithm"])
        .size()
        .rename("games")
        .reset_index()
    )
    orientation = (
        df.groupby(["algorithm_A", "algorithm_B", "player1_algorithm", "player2_algorithm", "winner_algorithm"])
        .size()
        .rename("games")
        .reset_index()
    )
    player_side = (
        df.assign(side=df["winner_player"].map({1: "Player 1", 2: "Player 2", -1: "Draw"}))
        .groupby(["algorithm_A", "algorithm_B", "side"])
        .size()
        .rename("games")
        .reset_index()
    )
    return overall, orientation, player_side

if RUN_LOCAL_TOURNAMENTS:
    start = time.time()
    all_results = []
    for idx, spec in enumerate(LOCAL_MATCHES):
        print(f"Running {DISPLAY[spec['a']]} vs {DISPLAY[spec['b']]}...")
        all_results.append(
            run_local_match(
                spec["a"],
                spec["b"],
                games=spec["games"],
                iterations=spec["iterations"],
                seed0=7000 + idx * 1000,
            )
        )
    df_local_results = pd.concat(all_results, ignore_index=True)
    output_path = DATA_DIR / "local_tournament_results.csv"
    df_local_results.to_csv(output_path, index=False)
    print(f"Saved local tournament rows to {output_path.relative_to(ROOT)}")
    print(f"Elapsed: {time.time() - start:.1f}s")

    overall, orientation, player_side = summarize_local_tournaments(df_local_results)
    display(df_local_results)
    display(overall)
    display(player_side)
    display(orientation)
else:
    print("Local tournaments were not run. Set RUN_LOCAL_TOURNAMENTS = True and re-run this cell.")


Local tournaments were not run. Set RUN_LOCAL_TOURNAMENTS = True and re-run this cell.


### 8.3 ID3 Against Optimized Numba Engines

The PopOut decision-tree pipeline also contains 30-game evaluations of ID3 agents against optimized engines. These were saved in the existing notebook and require the Numba environment to reproduce.

| Model | Opponent | Opponent iterations | ID3 wins | ID3 losses | Draws | ID3 win rate |
|---|---|---:|---:|---:|---:|---:|
| ID3 with features | FlatNumbaMCTS | 100 | 16 | 14 | 0 | 53.3% |
| ID3 with features | FlatNumbaMCTS | 500 | 14 | 16 | 0 | 46.7% |
| ID3 with features | FlatNumbaMCTS | 1,000 | 12 | 18 | 0 | 40.0% |
| ID3 with features | FlatNumbaMCTS | 5,000 | 9 | 21 | 0 | 30.0% |
| ID3 with features | FlatNumbaMCTS | 10,000 | 14 | 16 | 0 | 46.7% |
| ID3 with features | FlatNumbaSolverMCTS | 1,000 | 11 | 19 | 0 | 36.7% |
| ID3 with features | FlatNumbaSolverMCTS | 5,000 | 13 | 17 | 0 | 43.3% |
| ID3 with features | FlatNumbaSolverMCTS | 10,000 | 15 | 15 | 0 | 50.0% |
| ID3 with features | FlatNumbaSolverMCTS | 50,000 | 15 | 15 | 0 | 50.0% |
| ID3 with features | FlatNumbaSolverMCTS | 100,000 | 17 | 13 | 0 | 56.7% |
| ID3 Raw | FlatNumbaMCTS | 100 | 24 | 6 | 0 | 80.0% |
| ID3 Raw | FlatNumbaMCTS | 500 | 17 | 13 | 0 | 56.7% |
| ID3 Raw | FlatNumbaMCTS | 1,000 | 11 | 19 | 0 | 36.7% |
| ID3 Raw | FlatNumbaMCTS | 5,000 | 12 | 18 | 0 | 40.0% |
| ID3 Raw | FlatNumbaMCTS | 10,000 | 12 | 18 | 0 | 40.0% |
| ID3 Raw | FlatNumbaSolverMCTS | 1,000 | 18 | 12 | 0 | 60.0% |
| ID3 Raw | FlatNumbaSolverMCTS | 5,000 | 15 | 15 | 0 | 50.0% |
| ID3 Raw | FlatNumbaSolverMCTS | 10,000 | 17 | 13 | 0 | 56.7% |
| ID3 Raw | FlatNumbaSolverMCTS | 50,000 | 15 | 15 | 0 | 50.0% |
| ID3 Raw | FlatNumbaSolverMCTS | 100,000 | 17 | 13 | 0 | 56.7% |

These results show that ID3 is not merely a toy model: it remains competitive against high-throughput MCTS variants. However, the non-monotonic win rates also show that tournament samples are stochastic and color-sensitive, so larger repeated evaluations would be needed for final statistical confidence.


## 9. First-Player Advantage

Several results point to a strong first-player advantage:

- In the ID3Agent vs ID3Raw saved 20-game tournament, every game was won by Player 1: ID3Agent won the 10 games where it started, and ID3Raw won the 10 games where it started.
- In the fresh local 10-game ID3Agent vs ID3Raw run, the same pattern repeated: Player 1 won 10/10 games.
- In the technical documentation, 100k-iteration strong-engine tournaments suggested that Player 1 wins almost every game, although the empty-board root was not fully proven even with very large node budgets.

Conclusion: aggregate win rate alone can be misleading. Every serious comparison in this project should alternate colors and report orientation-specific counts.


## 10. How to Reproduce the Comparisons

To regenerate the comparison results locally:

1. Run the setup/import cells at the top of this notebook.
2. Go to section **8.2 Local Reproducible Tournament Run**.
3. Set `RUN_LOCAL_TOURNAMENTS = True`.
4. Run the cell.
5. Use the generated tables and `data/generated/local_tournament_results.csv` in the final report.

For optimized Numba engines, use the project environment:

```bash
conda env create -f environment.yml
conda activate popout-ai
jupyter notebook notebooks/PopOut_AI_Extensive_Report.ipynb
```

Then add Numba agent keys to `LOCAL_MATCHES`, for example `flat_numba`, `numba_solver`, or `flat_numba_solver`.


In [14]:
# Optional: inspect the latest locally generated tournament file.
local_results_path = DATA_DIR / "local_tournament_results.csv"
if local_results_path.exists():
    df_local_saved = pd.read_csv(local_results_path)
    display(df_local_saved.head())
    display(summarize_local_tournaments(df_local_saved)[0])
else:
    print("No local tournament CSV found yet. Run section 8.2 with RUN_LOCAL_TOURNAMENTS = True first.")


No local tournament CSV found yet. Run section 8.2 with RUN_LOCAL_TOURNAMENTS = True first.


## 11. Testing and Verification

The repository contains a broad pytest suite covering:

- bitboard behavior and PopOut rules,
- game-state persistence,
- standard MCTS,
- MCTS-Solver,
- Numba rules/search/MCTS,
- ID3 learning and discretization,
- dataset generation,
- CLI parsing,
- GUI imports and integration behavior,
- performance-oriented checks.

Known environment note: the current local Python is 3.13 and does not have Numba installed. The project environment is defined in `environment.yml` and targets Python 3.10 with Numba. Optimized-engine tests and notebooks should be run in that environment:

```bash
conda env create -f environment.yml
conda activate popout-ai
pytest -q
```

A previous non-Numba-focused run reported 283 passing tests and 2 failures, where one was due to missing Numba in the active environment and the other was a pandas string dtype expectation mismatch in the discretizer test.


## 12. Strengths, Risks, and Improvements

### Strengths

- Efficient bitboard design with O(1) move and win operations.
- Full UCT MCTS implementation with multiple variants.
- Solver-style proof propagation with minimax distance.
- High-throughput Numba engines for large-budget search and dataset generation.
- Custom ID3 implementation with Iris and PopOut workflows.
- Large generated PopOut datasets with proven-label metadata and mirroring.
- CLI and GUI cover the required play modes.
- Strong empirical culture: benchmarks, tournaments, plots, tests, and notebooks.

### Risks

- The optimized engines require the correct Conda environment; without Numba, they cannot be imported.
- ID3Agent is a hybrid agent because it includes tactical safeguards before tree prediction; this must be explained honestly.
- Pop labels are rare, so tree performance on pop decisions depends on oversampling and proven examples.
- Some tournament results are sample-size limited and show strong first-player effects.
- The empty-board game has not been fully solved; current evidence suggests a first-player advantage but does not constitute a complete proof.

### Best next improvements

1. Add a transposition table with Zobrist hashing to the solver engines.
2. Re-run all final tests and notebooks in the official Conda environment.
3. Increase tournament repetitions and report confidence intervals.
4. Separate pure ID3 classifier evaluation from hybrid playable-agent evaluation in final presentation slides.
5. Improve defensive-tactic accuracy for `opp_wins_next` positions.


## 13. Final Assessment

This is a strong Artificial Intelligence project because it does more than implement the minimum required algorithms. It builds a complete PopOut system: efficient game engine, playable interfaces, adversarial search, solver extensions, optimized search, learned decision-tree policies, datasets, tests, and empirical evaluation.

The most important technical conclusion is the trade-off between online search and learned policy:

- MCTS is flexible and improves with more computation, but decision time grows with iteration budget.
- Solver MCTS can convert tactical certainty into formal proof, especially in near-terminal positions.
- Numba optimization makes high-budget search practical.
- ID3 converts expensive oracle decisions into very fast move predictions, and the tactical-feature version is highly competitive for real-time play.

The most important experimental conclusion is that PopOut comparisons must alternate first player and second player. Several results are strongly affected by who starts, so exact orientation counts are essential for honest analysis.
